In [1]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
from transformers import PreTrainedTokenizer

from pivotal_tokens.constants import get_data_dir, get_artifacts_dir
from pivotal_tokens.hf.loading import  load_tokenizer
from pivotal_tokens.repo import SampleRepo, DictRepo


# EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.2-qwen3-1.7b-sps-tokens-small-prob-threshold'
# EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.3-qwen3-1.7b-sps-tokens-small-prob-threshold'
EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.4-qwen3-1.7b-sps-tokens-small-prob-threshold'
REPO_DIR = EXP_DIR / 'data' / 'repo'
PIVOTAL_TOKENS_FILE = EXP_DIR / 'data' / 'pivotal_tokens.csv'

TRACES_FILE = get_artifacts_dir() / 'exp1.1.1-qwen3-1.7b-traces.csv'
# PROCESSED_PIVOTAL_TOKENS_FILE = get_artifacts_dir() / 'exp2.1.2-qwen3-1.7b-pivotal-tokens.csv'
PROCESSED_PIVOTAL_TOKENS_FILE = get_artifacts_dir() / 'exp2.1.4-qwen3-1.7b-pivotal-tokens.csv'

QWEN3_1_7B_MODEL_ID = 'Qwen/Qwen3-1.7B'

In [2]:
tokenizer = load_tokenizer(QWEN3_1_7B_MODEL_ID)

In [3]:
base_repo = DictRepo(dirpath=REPO_DIR)

In [4]:
traces_df = pd.read_csv(TRACES_FILE)
traces_df[:2]

,id,query,ground_truth,trace
0,5a8b57f25542995d1e6f1371,Were Scott Derrickson and Ed Wood of the same ...,yes,"<think>\nOkay, let's see. The question is whet..."
1,5a8db19d5542994ba4e3dd00,Are Local H and For Against both from the Unit...,yes,"<think>\nOkay, let's see. The question is aski..."


In [5]:
tokens_df = pd.read_csv(PIVOTAL_TOKENS_FILE)
tokens_df['token_ids'] = tokens_df['token_ids'].apply(json.loads)
tokens_df[:2]

,sample_id,span_id,token_ids,span_text,prob_before,prob_after,prob_delta,is_pivotal,pivotal_context,metadata
0,5a8db19d5542994ba4e3dd00,fc805f90-e736-11f0-801c-08bfb8a7c92d,[264],a,0.46875,0.40625,-0.06250,False,<|im_start|>system\nAnswer the following quest...,"{'id': '5a8db19d5542994ba4e3dd00', 'question':..."
1,5a8db19d5542994ba4e3dd00,feb657ba-e736-11f0-801c-08bfb8a7c92d,[4948],political,0.40625,0.93750,0.53125,True,<|im_start|>system\nAnswer the following quest...,"{'id': '5a8db19d5542994ba4e3dd00', 'question':..."


In [6]:
df = pd.merge(traces_df, tokens_df, left_on='id', right_on='sample_id', how='inner')
df[:2]

,id,query,ground_truth,trace,sample_id,span_id,token_ids,span_text,prob_before,prob_after,prob_delta,is_pivotal,pivotal_context,metadata
0,5a8db19d5542994ba4e3dd00,Are Local H and For Against both from the Unit...,yes,"<think>\nOkay, let's see. The question is aski...",5a8db19d5542994ba4e3dd00,fc805f90-e736-11f0-801c-08bfb8a7c92d,[264],a,0.46875,0.40625,-0.06250,False,<|im_start|>system\nAnswer the following quest...,"{'id': '5a8db19d5542994ba4e3dd00', 'question':..."
1,5a8db19d5542994ba4e3dd00,Are Local H and For Against both from the Unit...,yes,"<think>\nOkay, let's see. The question is aski...",5a8db19d5542994ba4e3dd00,feb657ba-e736-11f0-801c-08bfb8a7c92d,[4948],political,0.40625,0.93750,0.53125,True,<|im_start|>system\nAnswer the following quest...,"{'id': '5a8db19d5542994ba4e3dd00', 'question':..."


In [7]:
def get_init_success_prob(sample_id: str, base_repo: DictRepo) -> float | None:
    repo = SampleRepo(base_repo=base_repo, sample_id=sample_id)
    subdivisions = repo.list(path="subdivisions")

    init_prob = None
    for subdiv in subdivisions:
        data = repo.load(path="subdivisions", key=subdiv)
        if len(data['prefix']) == 0 and \
                data['full_seq'].startswith('<think>') and \
                data['full_seq'].endswith('</think>'):
            init_prob = data['prob_before']
            break
    
    return init_prob

In [8]:
df['prob_init'] = df['sample_id'].apply(lambda x: get_init_success_prob(x, base_repo))

In [9]:
df = df[df['is_pivotal'] == True].copy(deep=True)

In [10]:
len(df)

605

In [11]:
df.to_csv(PROCESSED_PIVOTAL_TOKENS_FILE, index=False)